# Лабораторна. CurrencyWise: курси валют через API НБУ

**Тривалість:** 60 хвилин

**Контекст.** Ви — розробник стартапу CurrencyWise, який робить застосунок для студентів, що планують навчання чи стажування за кордоном. Застосунок має показувати актуальні курси валют, історію курсу за останній місяць і підказувати, як змінювалася гривня. Офіційні курси встановлює Національний банк України, і він публікує їх не лише на сайті, а й через **API** — сервіс, який віддає дані у зручному для програми вигляді.

**Сервіс:** [API НБУ](https://bank.gov.ua/ua/open-data/api-dev) — безкоштовний відкритий API з офіційними курсами валют, без реєстрації чи ключа.

**План роботи:**

| Частина | Що робимо | Час |
|---|---|---|
| 0 | Підготовка | 3 хв |
| 1 | Що таке API і JSON | 8 хв |
| 2 | Перший запит: курси на сьогодні | 10 хв |
| 3 | Курс на конкретну дату | 10 хв |
| 4 | Історія курсів за місяць | 12 хв |
| 5 | Аналіз і збереження результатів | 14 хв |
| 6 | Підсумок | 3 хв |

**Що здати:** цей ноутбук із заповненим кодом і відповідями в комірках «Ваша відповідь», а також файли `rates.csv` і `rates_raw.json`.

**Оцінювання:** максимум 1.25 бала. Бали вказано біля кожного завдання. Якщо кілька завдань оцінюються разом, бали ставляться, лише коли виконано всі з них. Код, що падає з помилкою, і відповіді без пояснення не зараховуються.

## Частина 0. Підготовка (3 хв)

Нам знадобляться:
- `requests` — уже знайома бібліотека для запитів;
- `json` і `csv` — вбудовані модулі Python для роботи з форматами JSON і CSV;
- `time` — для пауз між запитами;
- `date` і `timedelta` з вбудованого модуля `datetime` — для роботи з датами (пояснимо в Частині 3).

Якщо `requests` не встановлено, розкоментуйте перший рядок.

In [1]:
# !pip install requests

import csv
import json
import time
from datetime import date, timedelta

import requests

print("Усе готово!")

Усе готово!


## Частина 1. Що таке API і JSON (8 хв)

### Сайт і API: у чому різниця

Коли ви відкриваєте сайт, сервер повертає **HTML** — текст для браузера, у якому дані перемішані з дизайном. Щоб дістати з нього потрібне, доводиться шукати теги й класи, як у лабораторних зі скрейпінгу.

**API** (Application Programming Interface) — це «вхід для програм». Ви так само надсилаєте запит за адресою, але сервер повертає **лише дані** в структурованому форматі, найчастіше **JSON**. Дизайну, тегів і класів немає, тож нічого не треба шукати й чистити.

Запит до API складається з:
- **endpoint** — адреси конкретної «функції» API, наприклад `https://bank.gov.ua/NBUStatService/v1/statdirectory/exchange` («дай курси валют»);
- **параметрів** — уточнень, що саме ми хочемо: яку валюту, на яку дату, в якому форматі. Параметри дописуються до адреси після `?`, а в `requests` їх зручно передавати словником `params`.

### Що таке JSON

JSON — це текстовий формат даних, який виглядає майже як словники й списки Python:

| JSON | Python |
|---|---|
| `{"key": "value"}` (об'єкт) | `dict` |
| `[1, 2, 3]` (масив) | `list` |
| `"text"` | `str` |
| `41.5`, `840` | `float`, `int` |
| `true` / `false` | `True` / `False` |
| `null` | `None` |

Функція `json.loads(текст)` перетворює рядок JSON на об'єкти Python, а `response.json()` робить те саме з відповіддю сервера.

Виконайте приклад. Це скорочена відповідь API НБУ (курси умовні). Зверніть увагу: на верхньому рівні тут **список**, а кожен його елемент — **словник** з даними однієї валюти.

In [2]:
text = """
[
  {"r030": 840, "txt": "Долар США", "rate": 41.25, "cc": "USD", "exchangedate": "03.11.2026"},
  {"r030": 978, "txt": "Євро", "rate": 47.80, "cc": "EUR", "exchangedate": "03.11.2026"},
  {"r030": 985, "txt": "Злотий", "rate": 11.02, "cc": "PLN", "exchangedate": "03.11.2026"}
]
"""

data = json.loads(text)
print(type(data), len(data))
print(type(data[0]))
print(data[0])
print(data[1]["txt"], data[1]["rate"])

<class 'list'> 3
<class 'dict'>
{'r030': 840, 'txt': 'Долар США', 'rate': 41.25, 'cc': 'USD', 'exchangedate': '03.11.2026'}
Євро 47.8


**Що означають ключі:**
- `cc` — літерний код валюти (USD, EUR, PLN);
- `txt` — назва валюти українською;
- `rate` — скільки гривень коштує 1 одиниця валюти;
- `exchangedate` — дата, на яку встановлено курс (у форматі `ДД.ММ.РРРР`);
- `r030` — числовий код валюти за міжнародним стандартом.

**Як читати вкладені дані.** Вираз `data[1]["rate"]` читається зліва направо: «у списку `data` візьми елемент з індексом 1 (це словник), а в ньому — значення ключа `rate`».

**Питання 1.1.** *(0.25 б. за 1.1–2.3)* Чим відповідь API відрізняється від HTML-сторінки з лабораторних зі скрейпінгу? Чому для програми API зручніший? Наведіть одну причину, чому дані все одно іноді доводиться скрейпити.

**Ваша відповідь:** 
Відповідь API - це переважно JSON. А сторінка - це HTML, який додатково потрібно парсити, шукати теги і класи, і тільки вже за ними ми отримаємо дані, які потрібно. API дає це нам все на блюдечку одразу структуровано, так ще й в JSON який легко інтергується в типи даних які іспують в Python як dict, list і так далі.
Причина, чому дані все одно іноді доводиться скрейпити - це те, що не всі сайти банально надають API.

**Завдання 1.2.** *(0.25 б. за 1.1–2.3)* Використовуючи `data` з прикладу вище:
1. виведіть для кожної валюти рядок на зразок `USD (Долар США): 41.25 грн`;
2. знайдіть у списку злотий (код `PLN`) і виведіть його курс.

Підказка до пункту 2: у списку немає «ключа», за яким можна одразу знайти потрібну валюту, тому доведеться пройти циклом по всіх елементах і перевірити `cc` кожного.

In [6]:
for d in data:
    foutput = "{} ({}): {} грн"
    print(foutput.format(d["cc"], d["txt"], d["rate"]))

for d in data:
    if d["cc"] == "PLN":
        print(d["rate"])
        break

USD (Долар США): 41.25 грн
EUR (Євро): 47.8 грн
PLN (Злотий): 11.02 грн
11.02


## Частина 2. Перший запит: курси на сьогодні (10 хв)

Тепер звернемося до справжнього API. Щоб отримати офіційні курси всіх валют на сьогодні, потрібен endpoint:
```
https://bank.gov.ua/NBUStatService/v1/statdirectory/exchange?json
```
Слово `json` у кінці — це параметр без значення, який каже API віддати відповідь у форматі JSON (без нього API НБУ поверне інший формат, XML). Ми одразу запишемо його в адресу, а інші параметри будемо передавати через `params`: `requests` сам допише їх через `&`.

**Завдання 2.1.** *(0.25 б. за 1.1–2.3)* Зробіть запит до цього endpoint і виведіть:
1. код відповіді та повну адресу запиту (`response.url`);
2. тип результату `response.json()` і кількість елементів у ньому;
3. перший елемент повністю.

In [15]:
NBU_URL = "https://bank.gov.ua/NBUStatService/v1/statdirectory/exchange?json"

response = requests.get(NBU_URL)
print(response.status_code, response.url)
today_rates = response.json()
print(type(today_rates), len(today_rates))
print(today_rates[0])

200 https://bank.gov.ua/NBUStatService/v1/statdirectory/exchange?json
<class 'list'> 45
{'r030': 12, 'txt': 'Алжирський динар', 'rate': 0.33283, 'cc': 'DZD', 'exchangedate': '09.11.2026', 'special': None}


**Завдання 2.2.** *(0.25 б. за 1.1–2.3)* Шукати валюту циклом щоразу незручно. Перетворіть список на **словник** `rates`, де ключ — код валюти (`cc`), а значення — курс (`rate`). Тоді курс будь-якої валюти можна отримати одразу: `rates["USD"]`.

Кроки:
1. Створіть порожній словник `rates`.
2. Пройдіться циклом по `today_rates` і для кожного елемента запишіть `rates[код] = курс`.
3. Виведіть курси USD, EUR і PLN, а також дату, на яку їх встановлено (вона однакова для всіх елементів, тож можна взяти з першого).

In [16]:
retes = {r["cc"]: r["rate"] for r in today_rates}

for r in today_rates:
    cc = r["cc"]
    if cc in ["USD", "EUR", "PLN"]:
        print(cc, r["rate"], r["exchangedate"])

USD 45.3206 09.11.2026
EUR 50.6455 09.11.2026
PLN 11.5252 09.11.2026


**Завдання 2.3.** *(0.25 б. за 1.1–2.3)* Що буде, якщо попросити неіснуючу валюту? API НБУ дозволяє отримати курс лише однієї валюти через параметр `valcode`. Зробіть два запити з `params`:
1. `{"valcode": "USD"}`;
2. `{"valcode": "XYZ"}` — такої валюти не існує.

Для кожного виведіть код відповіді та `response.json()`.

Який код відповіді ви отримали для неіснуючої валюти? Як API повідомляє, що нічого не знайдено? Чому через це в коді не можна писати просто `response.json()[0]`?

In [19]:
response1 = requests.get(NBU_URL, params={"valcode": "USD"})
response2 = requests.get(NBU_URL, params={"valcode": "XYZ"})

print("USD code:", response1.status_code)
print("XYZ code:", response2.status_code)

USD code: 200
XYZ code: 200


**Ваша відповідь:** 
Який код відповіді ви отримали для неіснуючої валюти? - 200
Як API повідомляє, що нічого не знайдено? - пустим списком
Чому через це в коді не можна писати просто response.json()[0] - бо відповідь може бути пустим списком

## Частина 3. Курс на конкретну дату (10 хв)

API НБУ вміє повертати курс не лише на сьогодні, а й на будь-яку дату в минулому. Для цього є параметр `date` у форматі `РРРРММДД`, тобто рік, місяць і день без розділювачів: `20260115` — 15 січня 2026 року.

### Дати в Python

Записувати дати рядками вручну незручно й ризиковано: легко помилитися з кількістю днів у місяці. У Python для дат є тип `date` з модуля `datetime`:
```python
d = date(2026, 1, 15)           # 15 січня 2026
today = date.today()            # сьогоднішня дата
d.strftime("%Y%m%d")            # '20260115' — дата → рядок у потрібному форматі
d - timedelta(days=1)           # 14 січня 2026 — попередній день
```
- `strftime` («string format time») перетворює дату на рядок за шаблоном: `%Y` — рік з 4 цифр, `%m` — місяць з 2 цифр, `%d` — день з 2 цифр;
- `timedelta(days=N)` — «проміжок у N днів». Його можна віднімати від дати й додавати до неї, і Python сам правильно перейде через межу місяця чи року.

Виконайте приклад:

In [21]:
d = date(2026, 3, 1)
print(d, d.strftime("%Y%m%d"))
print(d - timedelta(days=1))
print(date.today())

2026-03-01 20260301
2026-02-28
2026-11-06


**Питання 3.1.** *(0.25 б. за 3.1 + 3.2)* Що вивів рядок `d - timedelta(days=1)`? Чому це зручніше, ніж зменшувати число дня в рядку `"20260301"` на одиницю?

**Ваша відповідь:** 
2026-02-28.
Чому це зручніше, ніж зменшувати число дня в рядку "20260301" на одиницю? - бо воно автоматично враховує зсув місяців або років якщо це початок місяця або року.

**Завдання 3.2.** *(0.25 б. за 3.1 + 3.2)* Напишіть функцію `get_rate(code, day)`, яка повертає курс валюти `code` на дату `day` (об'єкт `date`) як `float`, або `None`, якщо API нічого не повернув.

Кроки:
1. Зробіть запит до `NBU_URL` з параметрами `valcode` і `date` (дату перетворіть на рядок через `strftime`).
2. Перевірте код відповіді через `raise_for_status()`.
3. Якщо список у відповіді порожній, поверніть `None`. Інакше поверніть `rate` першого елемента.

Перевірте функцію: курс USD на 1 січня 2026 року, курс EUR тиждень тому, курс неіснуючої валюти `XYZ` на сьогодні.

In [24]:
def get_rate(code, day):
    params = {
        "valcode": code,
        "date": day.strftime("%Y%m%d"),
    }
    
    response = requests.get(NBU_URL, params=params)
    rate = response.json()
    
    if rate:
        return float(rate[0]["rate"])
    else:
        return None


print(get_rate("USD", date(2026, 1, 1)))
print(get_rate("EUR", date.today() - timedelta(days=7)))
print(get_rate("XYZ", date.today()))   # None

42.3532
50.6796
None


## Частина 4. Історія курсів за місяць (12 хв)

Застосунок має показувати, як змінювалися курси USD, EUR і PLN за останні 30 днів. Можна було б викликати `get_rate` для кожної валюти й кожного дня, але це 3 × 30 = 90 запитів. Розумніше на кожну дату робити **один** запит без `valcode`: API поверне всі валюти, і з них ми візьмемо потрібні три. Так вийде лише 30 запитів.

**Завдання 4.1.** *(0.25 б. за 4.1 + 4.2)* Напишіть функцію `get_day_rates(day, codes)`, яка повертає словник з датою й курсами лише потрібних валют, наприклад:
```python
{"date": "2026-11-03", "USD": 41.25, "EUR": 47.8, "PLN": 11.02}
```
а також **сирий** список, який повернув API (знадобиться для збереження в Частині 5). Функція може повертати два значення: `return row, raw`. Викликають таку функцію так: `row, raw = get_day_rates(...)`.

Кроки:
1. Зробіть запит до `NBU_URL` лише з параметром `date`.
2. Створіть словник `row` з ключем `"date"` і датою у форматі `РРРР-ММ-ДД` (так `date` виводиться за замовчуванням: `str(day)`).
3. Пройдіться по відповіді й для валют, код яких є у списку `codes` (перевірка `in`), додайте в `row` пару «код → курс».
4. Поверніть `row` і сирий список.

In [28]:
CODES = ["USD", "EUR", "PLN"]

def get_day_rates(day, codes):
    params = {
        "date": day.strftime("%Y%m%d"),
    }
    
    response = requests.get(NBU_URL, params=params)
    rates = response.json()

    row = {
        "date": str(day)
    }

    for r in rates:
        cc = r["cc"]
        if cc in codes:
            row[cc] = r["rate"]

    return row, rates

row, raw = get_day_rates(date.today(), CODES)
print(row)
print(len(raw))

{'date': '2026-11-06', 'USD': 45.4432, 'EUR': 51.0406, 'PLN': 11.6239}
45


**Завдання 4.2.** *(0.25 б. за 4.1 + 4.2)* Зберіть курси за останні 30 днів, від найстарішої дати до сьогодні.

Кроки:
1. Створіть порожній список `history` і порожній словник `raw_history` (ключ — дата рядком, значення — сирий список від API).
2. Напишіть цикл `for i in range(29, -1, -1):` — він пройде числа 29, 28, …, 0. Для кожного `i` дата дорівнює `date.today() - timedelta(days=i)`. Так дати йтимуть від найстарішої до сьогоднішньої.
3. Для кожної дати отримайте курси через `get_day_rates`, додайте `row` в `history`, а сирий список — у `raw_history`.
4. Виводьте прогрес кожні 10 днів і робіть паузу `time.sleep(0.2)`.

**Навіщо пауза, якщо це API?** Публічні API обмежують кількість запитів за хвилину (**rate limit**), щоб один користувач не перевантажив сервер. Якщо перевищити ліміт, API може повернути код **429 Too Many Requests** або тимчасово заблокувати ваші запити.

Перевірте: скільки рядків у `history`? Чи для кожної дати є курси всіх трьох валют?

In [29]:
history = []
raw_history = {}

for i in range(29, -1, -1):
    day = date.today() - timedelta(days=i)
    row, raw = get_day_rates(day, CODES)
    history.append(row)
    raw_history[str(day)] = raw
    if i % 10 == 0:
        print("Пройшло 10 днів", len(history))
    time.sleep(0.2)

print(len(history))
print(history)

Пройшло 10 днів 10
Пройшло 10 днів 20
Пройшло 10 днів 30
30
[{'date': '2026-10-08', 'USD': 44.8639, 'EUR': 50.1569, 'PLN': 11.4481}, {'date': '2026-10-09', 'USD': 44.8526, 'EUR': 50.1385, 'PLN': 11.4567}, {'date': '2026-10-10', 'USD': 44.8526, 'EUR': 50.1385, 'PLN': 11.4567}, {'date': '2026-10-11', 'USD': 44.8526, 'EUR': 50.1385, 'PLN': 11.4567}, {'date': '2026-10-12', 'USD': 44.8894, 'EUR': 50.3138, 'PLN': 11.4813}, {'date': '2026-10-13', 'USD': 44.9567, 'EUR': 50.2418, 'PLN': 11.4536}, {'date': '2026-10-14', 'USD': 45.0802, 'EUR': 50.438, 'PLN': 11.5245}, {'date': '2026-10-15', 'USD': 44.9801, 'EUR': 50.3453, 'PLN': 11.5297}, {'date': '2026-10-16', 'USD': 44.8871, 'EUR': 50.3081, 'PLN': 11.5141}, {'date': '2026-10-17', 'USD': 44.8871, 'EUR': 50.3081, 'PLN': 11.5141}, {'date': '2026-10-18', 'USD': 44.8871, 'EUR': 50.3081, 'PLN': 11.5141}, {'date': '2026-10-19', 'USD': 44.8625, 'EUR': 50.4977, 'PLN': 11.5573}, {'date': '2026-10-20', 'USD': 45.0013, 'EUR': 50.5311, 'PLN': 11.5582}, {'da

## Частина 5. Аналіз і збереження результатів (14 хв)

Аналізуємо дані на чистому Python, як у лабораторних зі скрейпінгу: `max`/`min` з параметром `key`, list comprehension, `zip`.

**Завдання 5.1.** *(0.25 б. за 5.1 + 5.2)* Для долара (USD) знайдіть за місяць:
1. середній курс (округлений до двох знаків);
2. найвищий і найнижчий курс разом з **датою**, коли він був.

Підказка до пункту 2: `max(history, key=функція)` поверне весь рядок (словник з датою й курсами), якщо функція повертає курс USD з рядка.

In [32]:
usd_rates = [h["USD"] for h in history]
usd_avg_rate = round(sum(usd_rates) / len(usd_rates), 2)
max_history_rate = max(history, key=lambda h: h["USD"])
min_history_rate = min(history, key=lambda h: h["USD"])

print(usd_avg_rate)
print(max_history_rate["USD"], max_history_rate["date"])
print(min_history_rate["USD"], min_history_rate["date"])

45.17
45.666 2026-11-04
44.8526 2026-10-09


**Ваша відповідь:** 
45.17
45.666 2026-11-04
44.8526 2026-10-09

**Завдання 5.2.** *(0.25 б. за 5.1 + 5.2)* Застосунок має показати, наскільки змінилася кожна валюта за місяць і як часто курс зростав.

1. Для кожної валюти з `CODES` порахуйте зміну за місяць у відсотках: `(останній курс - перший курс) / перший курс * 100`, округлену до двох знаків.
2. Для USD порахуйте, скільки днів курс **зріс** порівняно з попереднім днем. Для цього треба порівняти кожен курс з наступним. Зручний прийом: `zip(usd_rates, usd_rates[1:])` створює пари «день — наступний день»: (1-й, 2-й), (2-й, 3-й), ... Залишається порахувати пари, де другий курс більший за перший.

Яка валюта змінилася найбільше? Що означає додатна зміна курсу для студента, який отримує стипендію в гривнях і планує витрачати її за кордоном?

In [40]:
codes_changes = {}
for code in CODES:
    first_rate = history[0][code]
    last_rate = history[-1][code]
    change = round((last_rate - first_rate) / first_rate * 100, 2)
    codes_changes[code] = change

usd_up_days = 0
for now, nex in zip(usd_rates, usd_rates[1:]):
    if nex > now:
        usd_up_days += 1

print(codes_changes)
print(usd_up_days)

{'USD': 1.29, 'EUR': 1.76, 'PLN': 1.54}
11


**Ваша відповідь:** 
1. {'USD': 1.29, 'EUR': 1.76, 'PLN': 1.54}
2. 11

**Завдання 5.3.** *(0.25 б. за 5.3 + 5.4 + 6.1–6.2)* Збережіть список `history` у файл `rates.csv` через `csv.DictWriter` (як у лабораторних зі скрейпінгу). Колонки: `date`, `USD`, `EUR`, `PLN`. Підказка: назви колонок можна не писати вручну — `["date"] + CODES`.

In [41]:
with open("rates.csv", "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=["date"] + CODES)
    writer.writeheader()
    writer.writerows(history)

**Завдання 5.4.** *(0.25 б. за 5.3 + 5.4 + 6.1–6.2)* Збережіть **сирі** дані `raw_history` у файл `rates_raw.json`, а потім прочитайте його назад і перевірте, що дані збігаються.

Для запису й читання JSON-файлів є функції модуля `json`:
```python
with open("файл.json", "w", encoding="utf-8") as f:
    json.dump(дані, f, ensure_ascii=False, indent=2)

with open("файл.json", encoding="utf-8") as f:
    дані = json.load(f)
```
- `ensure_ascii=False` — зберегти українські літери як є (назви валют), а не як коди на кшталт `\u0414`;
- `indent=2` — гарні відступи, щоб файл було зручно читати людині.

Зверніть увагу: `json.dump` / `json.load` (без `s`) працюють з **файлами**, а `json.dumps` / `json.loads` (з `s` — string) — з **рядками**.

Відкрийте збережений файл. Чому сирі дані зберігають у JSON, а не в CSV? Навіщо взагалі зберігати сирі дані, якщо в CSV уже є все потрібне?

In [43]:
with open("rates_raw.json", "w", encoding="utf-8") as f:
    json.dump(raw_history, f, ensure_ascii=False, indent=2)

with open("rates_raw.json", encoding="utf-8") as f:
    raw_history = json.load(f)

**Ваша відповідь:** 

## Частина 6. Підсумок (3 хв)

**6.1.** *(0.25 б. за 5.3 + 5.4 + 6.1–6.2)* Як API НБУ повідомляє, що валюту не знайдено? Чим це відрізняється від звичайної помилки на кшталт 404? Чому важливо читати документацію API (або перевіряти на практиці), як саме конкретний API повідомляє про помилки?

**6.2.** *(0.25 б. за 5.3 + 5.4 + 6.1–6.2)* Застосунком CurrencyWise щодня користуватимуться 10 000 студентів, і кожен відкриває графік курсів за місяць. Скільки запитів до API НБУ це створить, якщо кожен користувач завантажуватиме історію сам? Як цього уникнути? (Підказка: курс за вчора вже ніколи не зміниться.)

**Ваша відповідь:** 

## Бонус (якщо залишився час) · не оцінюється

**Б.1.** Напишіть конвертер: користувач вводить через `input()` суму в гривнях і код валюти, а програма виводить, скільки валюти можна отримати за сьогоднішнім офіційним курсом. Не забудьте випадок, коли валюту не знайдено.

**Б.2.** Виведіть історію курсу USD за місяць у вигляді простої текстової діаграми: для кожної дати — рядок з символів `#`, довжина якого пропорційна різниці між курсом цього дня і мінімальним курсом місяця (наприклад, 1 `#` на кожні 0.02 грн).

In [ ]:
# Бонус